# PI3 Grupo 4 — Validação completa: binWidth=25 HU como configuração oficial

**Objetivo:** aplicar à configuração `binWidth=25` (comparável a Haarburger et al.,
2020, e alinhada à recomendação de van Timmeren et al., 2020, para modalidades
de escala física absoluta como a TC) a mesma bateria de validação que confirmou
o determinismo da Rodada 2 (`binCount=64`): extração completa, checagem de
qualidade e teste de determinismo por dupla execução com comparação numérica
exata.

### Por que repetir a validação inteira, e não só trocar o parâmetro

As duas correções que tornaram a v2 determinística — isolar o extractor por
nódulo e expandir a caixa envolvente antes da reamostragem — já estão
incorporadas neste notebook desde o início, porque não são específicas de
`binCount`; são do pipeline como um todo. Ainda assim, a validação completa é
repetida aqui porque a v2 foi validada nessas condições, não nestas: mudar
o parâmetro de discretização altera o histograma de intensidade calculado em
cada nódulo, e não há garantia a priori de que o mesmo comportamento
determinístico se mantenha sob uma distribuição de faixas diferente.

### Relação com o notebook de comparação já existente

O notebook `PI3_G4_comparacao_binCount_binWidth.ipynb` já extraiu esta mesma
amostra em `binWidth=25` uma única vez, para fins de comparação com a v2. Este
notebook não reaproveita aquele resultado — ele reextrai do zero e roda a
bateria de validação completa, incluindo o teste de determinismo por dupla
execução, que aquele notebook de comparação não fez.

**Nenhum resultado deste notebook foi obtido até você executá-lo.**

## 1. Instalação

Mesmo commit fixado do PyRadiomics usado em todas as rodadas anteriores, para
que a comparação entre configurações nunca seja confundida com diferença de
versão de biblioteca.

In [ ]:
PYRADIOMICS_COMMIT = '8ed579383'

!pip install git+https://github.com/AIM-Harvard/pyradiomics.git@{PYRADIOMICS_COMMIT}
!pip install pylidc
!pip install SimpleITK
!pip install idc-index
!pip install pandas pyarrow

In [ ]:
compat_src = '''
"""compat.py -- restaura APIs removidas, exigidas pelo pylidc 0.2.3."""
import configparser
import numpy as np

if not hasattr(configparser, "SafeConfigParser"):
    configparser.SafeConfigParser = configparser.ConfigParser

for _n, _t in {"float": float, "int": int, "bool": bool,
               "object": object, "str": str, "complex": complex}.items():
    if not hasattr(np, _n):
        setattr(np, _n, _t)

if not hasattr(np, "in1d"):     np.in1d = np.isin
if not hasattr(np, "alltrue"):  np.alltrue = np.all
if not hasattr(np, "sometrue"): np.sometrue = np.any
'''

import sys
with open('/content/compat.py', 'w') as fh:
    fh.write(compat_src)
sys.path.insert(0, '/content')
import compat  # noqa: F401

import numpy as np
print('NumPy', np.__version__, '| Python', sys.version.split()[0])

## 2. Drive e caminhos

In [ ]:
import os, glob, json, shutil, tarfile, time
import pandas as pd
from collections import Counter
from google.colab import drive

drive.mount('/content/drive')

BASE     = '/content/drive/MyDrive/PI3_Grupo4'
ARQUIVOS = f'{BASE}/dicom_tar'
FEATURES = f'{BASE}/features'
CONFIG   = f'{BASE}/config'
for d in (BASE, ARQUIVOS, FEATURES, CONFIG):
    os.makedirs(d, exist_ok=True)

LIDC_ROOT = '/content/lidc'
os.makedirs(LIDC_ROOT, exist_ok=True)
!df -h /content | tail -1

## 3. Parâmetros — binWidth=25 como configuração oficial

Idênticos à v2 em tudo, exceto o modo de discretização. Nome de versão próprio
(`piloto_v3_oficial`) para não colidir com os artefatos já salvos pela v2 nem
pelo notebook de comparação.

In [ ]:
PARAMS = {
    'versao_config': 'piloto_v3_oficial',

    'espacamento': [1.0, 1.0, 1.0],
    'interpolador': 'sitkBSpline',
    'modo_discretizacao': 'binWidth',    # configuracao oficial adotada
    'bin_count': 64,                      # mantido apenas como registro historico
    'bin_width': 25,                      # HU -- comparavel a Haarburger et al. (2020)

    'mascara': 'consenso50',
    'min_anotadores': 3,
    'diametro_min_mm': 3.0,

    'regra_rotulo': 'mediana',
    'mediana_max_benigno': 2.0,             # protocolo Sec. 5.2: mediana <= 2.0 -> alvo 0
    'mediana_min_maligno': 4.0,             # protocolo Sec. 5.2: mediana >= 4.0 -> alvo 1

    'n_pacientes_piloto': 25,
    'arquivar_tar_no_drive': True,
    'max_tentativas_extracao': 2,
    'margem_contexto_voxels': 4,
}

CLASSES = ['shape', 'firstorder', 'glcm', 'glrlm', 'glszm']

for k, v in PARAMS.items():
    print(f'{k:28s} {v}')

## 4. Obter os dados

In [ ]:
def inspecionar():
    tars = sorted(glob.glob(f'{ARQUIVOS}/*.tar'))
    dirs = [d for d in os.listdir(LIDC_ROOT)
            if os.path.isdir(os.path.join(LIDC_ROOT, d))] if os.path.exists(LIDC_ROOT) else []
    n_dcm = len(glob.glob(f'{LIDC_ROOT}/**/*.dcm', recursive=True))
    print(f'{len(tars)} tars no Drive | {len(dirs)} pastas locais | {n_dcm} arquivos .dcm')
    return tars, dirs, n_dcm

tars, dirs_existentes, n_dcm = inspecionar()
N = PARAMS['n_pacientes_piloto']

if n_dcm > 0 and len(dirs_existentes) >= N:
    FONTE = 'ja_pronto'
elif tars:
    FONTE = 'tar_drive'
else:
    FONTE = 'idc'
print('fonte:', FONTE)

In [ ]:
if FONTE == 'ja_pronto':
    print('DICOM já local; nada a fazer.')
elif FONTE == 'tar_drive':
    shutil.rmtree(LIDC_ROOT, ignore_errors=True); os.makedirs(LIDC_ROOT, exist_ok=True)
    for t in tars[:N]:
        with tarfile.open(t) as tf:
            tf.extractall(LIDC_ROOT)
    print(f'{min(N, len(tars))} tars descompactados')
elif FONTE == 'idc':
    from idc_index import IDCClient
    client = IDCClient.client()
    inv = client.sql_query("""
        SELECT PatientID, SeriesInstanceUID, Modality, series_size_MB
        FROM index WHERE collection_id = 'lidc_idri' AND Modality = 'CT'
    """)
    alvo = sorted(inv.PatientID.unique())[:N]
    shutil.rmtree(LIDC_ROOT, ignore_errors=True); os.makedirs(LIDC_ROOT, exist_ok=True)
    client.download_from_selection(patientId=alvo, downloadDir=LIDC_ROOT,
        dirTemplate='%PatientID/%StudyInstanceUID/%SeriesInstanceUID')

pids_disco = sorted(d for d in os.listdir(LIDC_ROOT)
                    if os.path.isdir(os.path.join(LIDC_ROOT, d)))
n_dcm = len(glob.glob(f'{LIDC_ROOT}/**/*.dcm', recursive=True))
print(f'{len(pids_disco)} pacientes | {n_dcm} arquivos .dcm')
if n_dcm == 0:
    raise RuntimeError('Nenhum .dcm em disco.')

## 5. Importar pylidc

In [ ]:
RC = os.path.expanduser('~/.pylidcrc')
with open(RC, 'w') as fh:
    fh.write(f'[dicom]\npath = {LIDC_ROOT}\nwarn = True\n')

for m in [m for m in list(sys.modules) if m.split('.')[0] == 'pylidc']:
    del sys.modules[m]

import pylidc as pl
from pylidc.utils import consensus

scans_teste = pl.query(pl.Scan).filter(pl.Scan.patient_id.in_(pids_disco)).all()
v = scans_teste[0].to_volume()
print(f'OK -- {scans_teste[0].patient_id}: volume {v.shape}')

## 6. Configuração do PyRadiomics

In [ ]:
import logging, radiomics
from radiomics import featureextractor
radiomics.logger.setLevel(logging.ERROR)

CFG = {'resampledPixelSpacing': PARAMS['espacamento'],
       'interpolator': PARAMS['interpolador'],
       'label': 1,
       'binWidth': PARAMS['bin_width']}

_checagem = featureextractor.RadiomicsFeatureExtractor(**CFG)
_checagem.disableAllFeatures()
for c in CLASSES:
    _checagem.enableFeatureClassByName(c)
print('configuração válida:', CFG)
del _checagem

MANIFESTO = {'params_grupo': PARAMS, 'config_pyradiomics': CFG,
             'classes_habilitadas': CLASSES,
             'pyradiomics_commit_fixado': PYRADIOMICS_COMMIT,
             'versao_pyradiomics_resolvida': radiomics.__version__,
             'versao_numpy': np.__version__,
             'gerado_em': time.strftime('%Y-%m-%d %H:%M:%S')}

with open(f"{CONFIG}/config_{PARAMS['versao_config']}.json", 'w') as fh:
    json.dump(MANIFESTO, fh, indent=2, ensure_ascii=False)

print(json.dumps(MANIFESTO, indent=2, ensure_ascii=False))

## 7. Extração — mesma lógica robusta da v2

Extractor isolado por nódulo, margem de contexto antes da reamostragem,
checagem de finitude e de faixa Hounsfield, retry com objeto limpo. Idêntico
ao que validou o determinismo da Rodada 2.

In [ ]:
import SimpleITK as sitk

CLEVEL = {'consenso50': 0.5, 'uniao': 0.01, 'intersecao': 1.0}


def montar_mascara(anns, modo):
    if modo in CLEVEL:
        cmask, cbbox, _ = consensus(anns, clevel=CLEVEL[modo])
        return cmask, cbbox
    if modo.startswith('leitor'):
        i = int(modo.replace('leitor', ''))
        if i >= len(anns):
            return None, None
        cmask, cbbox, _ = consensus([anns[i]], clevel=0.5)
        return cmask, cbbox
    raise ValueError(modo)


def rotular(escores, p):
    # retorna (valor_central, alvo, exclusion_reason)
    if p['regra_rotulo'] == 'mediana':
        c = float(np.median(escores))
    elif p['regra_rotulo'] == 'media':
        c = float(np.mean(escores))
    else:
        c = float(Counter(escores).most_common(1)[0][0])
    # Protocolo oficial (docs/protocolo_coorte_target_sprint2.md, Secao 5.2 e 5.3):
    #   mediana <= 2.0            -> alvo 0,    'included'
    #   mediana >= 4.0            -> alvo 1,    'included'
    #   mediana == 3.0            -> alvo None, 'consensus_indeterminate'
    #   mediana 2.5 ou 3.5        -> alvo None, 'fractional_median_even_raters'
    if c <= p['mediana_max_benigno']:
        return c, 0, 'included'
    if c >= p['mediana_min_maligno']:
        return c, 1, 'included'
    if abs(c - 3.0) < 1e-9:
        return c, None, 'consensus_indeterminate'
    return c, None, 'fractional_median_even_raters'


def expandir_bbox(cbbox, vol_shape, margem):
    novo = []
    for eixo, tam in zip(cbbox, vol_shape):
        ini = max(0, eixo.start - margem)
        fim = min(tam, eixo.stop + margem)
        novo.append(slice(ini, fim))
    return tuple(novo)


def extrair_features_isolado(img, msk, cfg, classes):
    ext = featureextractor.RadiomicsFeatureExtractor(**cfg)
    ext.disableAllFeatures()
    for c in classes:
        ext.enableFeatureClassByName(c)
    return ext.execute(img, msk)


def extrair_com_retry(img, msk, cfg, classes, tentativas):
    erro = None
    for _ in range(tentativas):
        try:
            return extrair_features_isolado(img, msk, cfg, classes)
        except Exception as e:
            erro = e
    raise erro


def extrair_scan(scan, p, cfg, classes):
    vol = scan.to_volume()
    esp = [float(scan.pixel_spacing), float(scan.pixel_spacing), float(scan.slice_spacing)]

    linhas, descartes = [], []
    for idx, anns in enumerate(scan.cluster_annotations()):
        nid = f'{scan.patient_id}_N{idx:02d}'
        if len(anns) < p['min_anotadores']:
            descartes.append((nid, 'leitores_insuficientes')); continue
        diams = [float(a.diameter) for a in anns]
        if np.mean(diams) < p['diametro_min_mm']:
            descartes.append((nid, 'diametro_abaixo_do_minimo')); continue
        escores = [int(a.malignancy) for a in anns]
        central, alvo, motivo_alvo = rotular(escores, p)
        # indeterminados NAO sao descartados (protocolo, Secao 8, item 2): ficam
        # na tabela com alvo=None e exclusion_reason preenchido
        try:
            cmask, cbbox = montar_mascara(anns, p['mascara'])
            if cmask is None or cmask.sum() == 0:
                descartes.append((nid, 'mascara_vazia')); continue

            cbbox_exp = expandir_bbox(cbbox, vol.shape, p['margem_contexto_voxels'])
            sub = vol[cbbox_exp]
            mask_exp = np.zeros(sub.shape, dtype=cmask.dtype)
            offset = tuple(a.start - b.start for a, b in zip(cbbox, cbbox_exp))
            slices_orig = tuple(slice(o, o + s) for o, s in zip(offset, cmask.shape))
            mask_exp[slices_orig] = cmask

            valores = sub[mask_exp.astype(bool)]
            if not np.all(np.isfinite(valores)):
                descartes.append((nid, 'intensidade_nao_finita')); continue
            if np.any(np.abs(valores) > 5000):
                descartes.append((nid, 'intensidade_fora_de_faixa_hu')); continue

            img = sitk.GetImageFromArray(np.transpose(sub, (2, 0, 1)).astype(np.float32))
            msk = sitk.GetImageFromArray(np.transpose(mask_exp.astype(np.uint8), (2, 0, 1)))
            img.SetSpacing(esp); msk.SetSpacing(esp)

            feats = extrair_com_retry(img, msk, cfg, classes, p['max_tentativas_extracao'])
        except Exception as e:
            descartes.append((nid, f'erro_extracao:{type(e).__name__}')); continue

        linha = {'nodule_id': nid, 'patient_id': scan.patient_id,
                 'scan_id': int(scan.id), 'nodule_idx': idx,
                 'n_anotadores': len(anns),
                 'malignancy_escores': '|'.join(map(str, escores)),
                 'malignancy_mediana': float(np.median(escores)),
                 'malignancy_media': float(np.mean(escores)),
                 'malignancy_moda': float(Counter(escores).most_common(1)[0][0]),
                 'diametro_medio_mm': float(np.mean(diams)),
                 'voxels_mascara': int(cmask.sum()),
                 'valor_central': central, 'alvo': alvo,
                 'indeterminado': alvo is None,
                 'exclusion_reason': motivo_alvo,
                 'config': p['versao_config'],
                 'mascara': p['mascara']}
        linha.update({k: v2 for k, v2 in feats.items() if not k.startswith('diagnostics')})
        linhas.append(linha)
    return linhas, descartes


print('funções definidas')

## 8. Execução principal (25 pacientes)

In [ ]:
todas_linhas, todos_descartes, falhas = [], [], []
t0 = time.time()

for k, pid in enumerate(pids_disco, 1):
    for scan in pl.query(pl.Scan).filter(pl.Scan.patient_id == pid).all():
        try:
            L, D = extrair_scan(scan, PARAMS, CFG, CLASSES)
            todas_linhas.extend(L); todos_descartes.extend(D)
        except Exception as e:
            falhas.append((pid, f'{type(e).__name__}: {e}'))
    if k % 5 == 0 or k == len(pids_disco):
        print(f'{k}/{len(pids_disco)} | {len(todas_linhas)} nódulos | '
              f'{(time.time()-t0)/k:.1f}s/paciente')

print(f'\nextraídos  : {len(todas_linhas)}')
print(f'descartados: {len(todos_descartes)}')
print(f'falhas     : {len(falhas)}')
if todos_descartes:
    print('motivos:', dict(Counter(d[1] for d in todos_descartes)))

if not todas_linhas:
    raise RuntimeError('Nenhum nódulo extraído — verificar parâmetros e ambiente.')

## 9. Tabela derivada

In [ ]:
df = pd.DataFrame(todas_linhas)

META = ['nodule_id', 'patient_id', 'scan_id', 'nodule_idx', 'n_anotadores',
        'malignancy_escores', 'malignancy_mediana', 'malignancy_media',
        'malignancy_moda', 'diametro_medio_mm', 'voxels_mascara',
        'valor_central', 'alvo', 'indeterminado', 'exclusion_reason',
        'config', 'mascara']
FEAT = [c for c in df.columns if c not in META]
df = df[META + sorted(FEAT)]
for c in FEAT:
    df[c] = pd.to_numeric(df[c], errors='coerce').astype('float64')

sufixo = f"{PARAMS['versao_config']}_{PARAMS['mascara']}"
df.to_csv(f'{FEATURES}/piloto_{sufixo}.csv', index=False)
df.to_parquet(f'{FEATURES}/piloto_{sufixo}.parquet', index=False)
if todos_descartes:
    pd.DataFrame(todos_descartes, columns=['nodule_id', 'motivo']).to_csv(
        f'{FEATURES}/descartes_{sufixo}.csv', index=False)

print(f'{df.shape[0]} linhas x {df.shape[1]} colunas ({len(META)} metadados, {len(FEAT)} features)')
df.head(3)

## 10. Validação de qualidade

In [ ]:
print('=== 10.1 CONTAGEM POR CLASSE ===')
por_classe = {}
for c in CLASSES:
    cols = [x for x in FEAT if f'_{c}_' in x]
    por_classe[c] = len(cols)
    print(f'  {c:12s} {len(cols):3d}')
print(f'  {"TOTAL":12s} {sum(por_classe.values()):3d}')

print('\n=== 10.2 AUSENTES E DEGENERADAS ===')
na = df[FEAT].isna().sum(); na = na[na > 0].sort_values(ascending=False)
print(f'  colunas com NaN: {len(na)}')
inf = {c: int(np.isinf(df[c]).sum()) for c in FEAT if np.isinf(df[c]).any()}
print(f'  colunas com inf: {len(inf)}')
const = [c for c in FEAT if df[c].nunique(dropna=True) <= 1]
print(f'  colunas constantes: {len(const)}')

print('\n=== 10.3 SANIDADE FÍSICA ===')
ec = [c for c in FEAT if c.endswith('shape_Sphericity')]
if ec:
    e = df[ec[0]]
    fora = int(((e < 0) | (e > 1)).sum())
    print(f'  Sphericity fora de [0,1]: {fora}')
print(f'  voxels na máscara: mín {df.voxels_mascara.min()} | máx {df.voxels_mascara.max()}')

print('\n=== 10.4 DISTRIBUIÇÃO DO RÓTULO ===')
n3 = int(df.indeterminado.sum())
print(f'  indeterminados: {n3} ({100*n3/len(df):.1f}%)')
print(f'  alvo=0: {int((df.alvo==0).sum())} | alvo=1: {int((df.alvo==1).sum())}')

## 11. Teste de determinismo — dupla execução

O mesmo procedimento que validou a v2: roda a extração da amostra de 10
pacientes duas vezes de forma independente e compara contagem, identificadores
e valores numéricos de todas as features.

In [ ]:
N_TESTE = min(10, len(pids_disco))
amostra_teste = pids_disco[:N_TESTE]


def rodar_amostra():
    linhas = []
    for pid in amostra_teste:
        for scan in pl.query(pl.Scan).filter(pl.Scan.patient_id == pid).all():
            L, _ = extrair_scan(scan, PARAMS, CFG, CLASSES)
            linhas.extend(L)
    return pd.DataFrame(linhas)


print(f'Rodando 2x sobre {N_TESTE} pacientes (binWidth={PARAMS["bin_width"]})...')
r1 = rodar_amostra()
r2 = rodar_amostra()

print(f'rodada 1: {len(r1)} nódulos | rodada 2: {len(r2)} nódulos')
print(f'mesma contagem: {len(r1) == len(r2)}')

In [ ]:
ids1 = set(r1['nodule_id']) if len(r1) else set()
ids2 = set(r2['nodule_id']) if len(r2) else set()

print('mesmos IDs:', ids1 == ids2)
print('só na execução 1:', ids1 - ids2)
print('só na execução 2:', ids2 - ids1)

In [ ]:
if ids1 == ids2 and len(r1):
    df1 = r1.set_index('nodule_id').sort_index()
    df2 = r2.set_index('nodule_id').sort_index()

    cols_num = [c for c in df1.columns
                if pd.api.types.is_numeric_dtype(df1[c]) and df1[c].dtype != bool]

    diff = (df1[cols_num] - df2[cols_num]).abs()
    max_diff = diff.max().max()
    print(f'maior diferença absoluta entre as duas execuções: {max_diff:.2e}')
    determinismo_confirmado = bool(max_diff < 1e-6)
    print('determinístico' if determinismo_confirmado else 'ATENÇÃO: ainda há diferença numérica')
else:
    max_diff = float('nan')
    determinismo_confirmado = False
    print('ATENÇÃO: IDs divergem — comparação numérica não realizada')

## 12. Relatório final e exportação

In [ ]:
# rastreabilidade do protocolo, Secao 5.3: as duas causas de indefinicao de
# alvo sao contadas a partir de exclusion_reason e reportadas separadamente
n_consensus_indeterminate = int(
    (df.exclusion_reason == 'consensus_indeterminate').sum())
n_fractional_median = int(
    (df.exclusion_reason == 'fractional_median_even_raters').sum())
n_indeterminado_col = int(df['indeterminado'].sum())
assert n3 == n_consensus_indeterminate + n_fractional_median, (
    f'indeterminados ({n3}) != consensus_indeterminate '
    f'({n_consensus_indeterminate}) + fractional_median_even_raters '
    f'({n_fractional_median})')
assert n3 == n_indeterminado_col, (
    f'indeterminados ({n3}) != soma da coluna indeterminado '
    f'({n_indeterminado_col})')

relatorio = {
    'config': MANIFESTO,
    'execucao_principal': {
        'pacientes': len(pids_disco),
        'pacientes_com_nodulo': int(df.patient_id.nunique()),
        'nodulos_extraidos': int(len(df)),
        'nodulos_descartados': len(todos_descartes),
        'motivos_descarte': dict(Counter(d[1] for d in todos_descartes)),
        'falhas': len(falhas),
        'features_por_classe': por_classe,
        'colunas_com_nan': int(len(na)),
        'colunas_com_inf': len(inf),
        'colunas_constantes': len(const),
        'indeterminados': n3,
        'consensus_indeterminate': n_consensus_indeterminate,
        'fractional_median_even_raters': n_fractional_median,
        'alvo_0': int((df.alvo == 0).sum()),
        'alvo_1': int((df.alvo == 1).sum()),
    },
    'teste_determinismo': {
        'pacientes_testados': N_TESTE,
        'nodulos_execucao_a': int(len(r1)),
        'nodulos_execucao_b': int(len(r2)),
        'mesmos_ids': bool(ids1 == ids2),
        'maior_diferenca_absoluta': float(max_diff) if not np.isnan(max_diff) else None,
        'determinismo_confirmado': determinismo_confirmado,
    },
}

with open(f'{FEATURES}/validacao_{sufixo}.json', 'w') as fh:
    json.dump(relatorio, fh, indent=2, ensure_ascii=False, default=str)

print(json.dumps({k: v for k, v in relatorio.items() if k != 'config'},
                 indent=2, ensure_ascii=False))
print(f'\nSalvo em features/validacao_{sufixo}.json')

---
### Como usar este resultado no relatório

Se `determinismo_confirmado: true` e os números de cobertura (Seção 8) forem
próximos aos da v2 (binCount=64) — o que é esperado, já que discretização não
deveria alterar quantos nódulos são elegíveis, apenas seus valores de
intensidade e textura — a configuração `binWidth=25` está validada com o mesmo
padrão de rigor da Rodada 2 e pode substituí-la como configuração oficial do
pipeline, preservando a v2 como registro de análise de sensibilidade (Seção
3.5 do relatório).

Se qualquer uma das verificações falhar, trate como um novo incidente: isole o
caso, identifique a causa raiz e não prossiga para a extração completa nos
1.010 pacientes antes de corrigir e revalidar — o mesmo processo já seguido
duas vezes na Rodada 2.